# Chapter 14 — Scientific Statistics with SciPy

SciPy adds established numerical algorithms and statistical procedures on top of NumPy. This chapter focuses on distributions, descriptive statistics, confidence intervals, hypothesis tests, correlation, and interpretation.

## Prerequisites
NumPy arrays, Pandas filtering/grouping, and foundational probability concepts.

## Learning goals
- Select and parameterize continuous and discrete probability distributions.
- Compute descriptive statistics and confidence intervals.
- Choose and interpret common hypothesis tests.
- Assess correlation and avoid common inferential errors.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Chapter map

1. SciPy architecture and stats module
2. Descriptive statistics
3. Probability distributions
4. Hypothesis testing
5. Proportions, association, and multiple testing
6. Confidence intervals and effect sizes

### SciPy architecture and stats module
SciPy is a collection of scientific-computing subpackages. `scipy.stats` provides distributions, summary statistics, tests, and correlation measures. A frozen distribution object stores parameters and exposes methods such as `pdf`, `pmf`, `cdf`, `ppf`, `rvs`, `mean`, and `var`. The `loc` and `scale` parameters shift and scale many continuous distributions; distribution-specific shape parameters define form.

#### Demonstration 1 — Summarize a numeric sample

Run the code and inspect both the output and the assumptions behind it.

In [2]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())
import pandas as pd, numpy as np
from scipy import stats
sales=pd.read_csv(DATA/"sales.csv"); x=sales.profit.to_numpy()
result=stats.describe(x); print(result); print("IQR:",stats.iqr(x),"Trimmed mean:",stats.trim_mean(x,.10))

Data directory: /mnt/data/python_data_analysis_book/data
DescribeResult(nobs=np.int64(240), minmax=(np.float64(238.21), np.float64(386426.57)), mean=np.float64(38969.336833333335), variance=np.float64(5858828228.084906), skewness=np.float64(2.885722984802496), kurtosis=np.float64(8.293687097937257))
IQR: 34120.1375 Trimmed mean: 18444.4840625


### Descriptive statistics
`stats.describe(a, axis=0, ddof=1, bias=True, nan_policy="propagate")` reports count, minimum/maximum, mean, variance, skewness, and kurtosis. `stats.iqr(a, axis=None, rng=(25,75), scale=1.0, nan_policy=...)` measures a quantile range. `stats.trim_mean(a, proportiontocut, axis=0)` removes equal tail proportions before averaging. Know whether a function uses sample or population variance and whether kurtosis is Fisher excess or Pearson.

#### Demonstration 2 — Explore a normal distribution

Run the code and inspect both the output and the assumptions behind it.

In [3]:
from scipy import stats
dist=stats.norm(loc=100,scale=15)
print("P(X <= 115):",dist.cdf(115)); print("P(X > 130):",dist.sf(130)); print("95th percentile:",dist.ppf(.95)); print("sample:",dist.rvs(size=5,random_state=42))

P(X <= 115): 0.8413447460685429
P(X > 130): 0.022750131948179195
95th percentile: 124.67280440427209
sample: [107.4507123   97.92603548 109.71532807 122.84544785  96.48769938]


### Probability distributions
Examples: `stats.norm(loc,scale)`, `stats.t(df,loc,scale)`, `stats.binom(n,p)`, `stats.poisson(mu)`, and `stats.expon(scale)`. `pdf` is a density, not point probability; `pmf` is a discrete point probability. `cdf(x)` gives cumulative probability; `sf(x)` is often more numerically stable for upper tails; `ppf(q)` is the quantile function. `rvs(size, random_state)` draws random samples.

#### Demonstration 3 — Compare two independent groups

Run the code and inspect both the output and the assumptions behind it.

In [4]:
import pandas as pd
from scipy import stats
sales=pd.read_csv(DATA/"sales.csv")
north=sales.loc[sales.region=="North","profit"]; south=sales.loc[sales.region=="South","profit"]
res=stats.ttest_ind(north,south,equal_var=False)
print(res); print("Group means:",north.mean(),south.mean())

TtestResult(statistic=np.float64(-0.4399179966447202), pvalue=np.float64(0.6608982364971381), df=np.float64(105.28573033827658))
Group means: 37319.01867924528 44162.28017241378


### Hypothesis testing
A test returns a statistic and p-value. A p-value is the probability, under the null and test assumptions, of data at least as extreme as observed; it is not the probability that the null is true. `ttest_1samp`, `ttest_ind`, and `ttest_rel` address one-sample, independent-group, and paired mean comparisons. `equal_var=False` in `ttest_ind` gives Welch’s test. Check independence, sampling design, distributional assumptions, and variance structure.

#### Demonstration 4 — Measure a correlation

Run the code and inspect both the output and the assumptions behind it.

In [5]:
from scipy import stats
r=stats.pearsonr(sales.units,sales.revenue); rho=stats.spearmanr(sales.units,sales.revenue)
print("Pearson:",r); print("Spearman:",rho)

Pearson: PearsonRResult(statistic=np.float64(0.41213031816893975), pvalue=np.float64(2.946244133590899e-11))
Spearman: SignificanceResult(statistic=np.float64(0.5663055981989009), pvalue=np.float64(9.36633332558205e-22))


### Proportions, association, and multiple testing
`chi2_contingency` tests independence in a contingency table; `fisher_exact` is useful for small 2x2 tables. `pearsonr` measures linear association and provides a test; `spearmanr` uses ranks and captures monotonic association. Multiple comparisons increase false-positive risk; adjust p-values or pre-specify comparisons. Statistical significance is distinct from practical importance and causal evidence.

#### Demonstration 5 — Build a t confidence interval

Run the code and inspect both the output and the assumptions behind it.

In [6]:
import numpy as np
from scipy import stats
x=sales.profit.dropna().to_numpy(); n=len(x); mean=x.mean(); se=stats.sem(x)
ci=stats.t.interval(.95,df=n-1,loc=mean,scale=se)
print({"n":n,"mean":mean,"95% CI":ci})

{'n': 240, 'mean': np.float64(38969.336833333335), '95% CI': (np.float64(29236.20446856826), np.float64(48702.469198098406))}


### Confidence intervals and effect sizes
A confidence interval is generated by a procedure with a stated long-run coverage level under assumptions. For a mean with unknown standard deviation, a t interval uses sample standard error and a t critical value. Report effect size, interval, sample size, and test assumptions rather than only a p-value.

## Self-check

Without looking back, explain the key terms, identify the important parameters, describe at least one alternative or variation, and state one limitation of the methods covered. Then answer the exercises below.

### Review questions

- Explain one important concept from: SciPy architecture and stats module.
- Explain one important concept from: Descriptive statistics.
- Explain one important concept from: Probability distributions.
- Explain one important concept from: Hypothesis testing.
- Explain one important concept from: Proportions, association, and multiple testing.